# ChadGPT — open, run, and test

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://huggingface.co/sam-eer12/chadGPT/colab)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://huggingface.co/sam-eer12/chadGPT/kaggle)

This notebook installs the required packages and automatically fetches the inference code, tokenizer, configuration, and model weights from **sam-eer12/chadGPT** on Hugging Face. No code to clone, files to upload, or Hugging Face login/token is needed.

1. Open either launcher link and sign in to Colab or Kaggle.
2. On **Kaggle**, enable **Internet** in notebook settings. Optionally select a GPU accelerator; CPU also works. Complete any account verification that Kaggle requests. On **Colab**, connect a runtime; a GPU is optional.
3. Run **Install dependencies → Download and load → Test inference** in order. Read or change the example prompt in the test cell to try the model inside the notebook.

The first model download is approximately **1 GB**. CUDA is selected when available, otherwise Apple MPS or CPU. Installation keeps the notebook's existing PyTorch/CUDA stack. No Hugging Face PRO subscription is required.

[Model card and full training history](https://huggingface.co/sam-eer12/chadGPT)

ChadGPT is an experimental 250M-parameter model trained from scratch on Kaggle over approximately two months. Its answers and generated code can be incorrect.

## 1. Install inference dependencies

Kaggle and Colab normally provide PyTorch already. This cell keeps the existing PyTorch/CUDA stack. On a local notebook without PyTorch, install it separately first.

In [ ]:
%pip install -q tiktoken==0.12.0 safetensors==0.8.0 huggingface_hub==1.21.0

## 2. Download and load the model

Download the standalone inference files and model weights from the public Hugging Face repository. The weights and inference implementation are pinned to the verified initial model release.

In [ ]:
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

MODEL_REPO = "sam-eer12/chadGPT"
MODEL_REVISION = "1d79625c1b3194140ccfcb933ae49b6599820fbb"

torch.set_num_threads(2)
print('Loading ChadGPT (approximately 1 GB on first download)…')
model_dir = Path(snapshot_download(
    repo_id=MODEL_REPO,
    revision=MODEL_REVISION,
    allow_patterns=[
        "model.safetensors", "config.json", "tokenizer.tiktoken",
        "tokenizer_config.json", "inference.py", "modeling_chadgpt.py",
    ],
))
sys.path.insert(0, str(model_dir.resolve()))
from inference import chat, load_model

model, tokenizer = load_model(model_dir, device="auto")
print('Model ready on:', next(model.parameters()).device)
print('PyTorch:', torch.__version__)
print('Run the next cell to test inference.')

## 3. Test inference inside the notebook

Run this cell for a sample response. Change **prompt** and run it again to ask another question. This test prints the answer here and does not start a public web server.

In [ ]:
prompt = "Explain what a neural network is in simple terms." #@param {type:"string"}
max_new_tokens = 96 #@param {type:"integer"}

response = chat(
    model, tokenizer,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_new_tokens=int(max_new_tokens), temperature=0.7, top_k=40,
)
print(response or "The model ended its response without generating text. Try another prompt.")